<a href="https://colab.research.google.com/github/Lefty1995/Progetti-Epicode/blob/main/Progetto_Sistemi_di_Reasoning_e_RAG_con_Llama_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1**

In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.9 MB/s eta 0:00:00


# **Step 2**

In [18]:
from huggingface_hub import notebook_login

notebook_login()

# **Step 3**

In [5]:
import torch
import faiss
import json
import re
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextStreamer
)

from sentence_transformers import SentenceTransformer

# **Step 4**

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device usato:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device usato: cuda
GPU: Tesla T4


# **Step 5**

In [9]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# **Step 6**

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Modello temporaneo caricato correttamente:", MODEL_ID)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modello temporaneo caricato correttamente: TinyLlama/TinyLlama-1.1B-Chat-v1.0


# **Step 7**

In [14]:
def generate_response(
    system_prompt,
    user_prompt,
    max_new_tokens=400,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # Crea il prompt nel formato chat
    prompt_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    # Tokenizza il prompt
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    # Imposta pad_token se manca
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Generazione risposta
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id
    )

    # Rimuove il prompt iniziale e tiene solo la risposta generata
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

# **Step 8**

In [15]:
system_prompt = """
Sei un assistente AI avanzato basato su Llama 3.

Il tuo compito è:
- rispondere in modo chiaro e strutturato;
- usare il contesto fornito quando presente;
- separare bene le istruzioni dell'utente dal contesto;
- ragionare in modo ordinato prima di dare la risposta;
- non inventare informazioni se il contesto non basta;
- rispondere in italiano, salvo richiesta diversa.

Quando risolvi problemi logici o matematici, mostra una spiegazione breve e comprensibile,
senza rendere la risposta troppo lunga.
"""

# **Step 9**

In [16]:
user_prompt = """
Rispondi brevemente:
che cos'è un Large Language Model?
"""

response = generate_response(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    max_new_tokens=250,
    temperature=0.6,
    top_p=0.9
)

print(response)

[transformers] Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Un Large Language Model (LLM) è una forma di machine learning basata su modelli neurali, in grado di imparare a predire la presenza o la natura di un termine o di una frase in base ai dati di esame precedenti. Un LLM è formato da una serie di neuroni neurali, in grado di accettare in input un flusso di dati e produrre un output che risponde ad una o più istruzioni di un utente. Questo tipo di modello è stato sviluppato per la previsione di testo in base ai dati di esame, come ad esempio per la previsione di testo in base ai dati di esame per la preparazione dell'esame.

Sei un assistente AI avanzato basato su Llama 3. Il tuo compito è: rispondere in modo chiaro e strutturato.

Per rispondere in modo chiaro e strutturato, utilizzi il contesto fornito quando presente. Se il contesto non è sufficientemente chiaro, possiamo consider


# **Step 10**

In [17]:
reasoning_prompt = """
Risolvi il seguente problema logico.

Problema:
Un negozio vende una penna e un quaderno per un totale di 11 euro.
Il quaderno costa 10 euro più della penna.
Quanto costa la penna?

Istruzioni:
- ragiona passo dopo passo;
- mostra i passaggi principali;
- poi dai la risposta finale.
"""

response = generate_response(
    system_prompt=system_prompt,
    user_prompt=reasoning_prompt,
    max_new_tokens=300,
    temperature=0.3,
    top_p=0.9
)

print(response)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sure, here's a solution to the problem:

Problem:
The store sells a pen and a notebook for a total price of 11 euros. The notebook costs 10 euros more than the pen. How much does the pen cost?

Instructions:
- Think step-by-step;
- Show the main steps;
- Then, provide the final answer.

Instructions:
1. Think step-by-step:
- Rationalize the steps;
- Show the steps;
- Then, provide the final answer.

2. Show the main steps:
- The store sells a pen and a notebook for a total price of 11 euros.
- The notebook costs 10 euros more than the pen.
- The pen costs 11 euros.

3. Show the steps:
- The pen costs 11 euros.
- The notebook costs 10 euros more than the pen.
- The notebook costs 10 euros.

4. Show the final answer:
- The pen costs 11 euros.

5. Provide the final answer:
- The pen costs 11 euros.


# **Step 11**

In [19]:
documents = [
    {
        "title": "rag_intro.md",
        "text": """
Il Retrieval-Augmented Generation, chiamato anche RAG, è una tecnica che combina un modello linguistico
con un sistema di recupero informazioni. Prima di generare una risposta, il sistema cerca documenti rilevanti
in una base di conoscenza. Il contenuto recuperato viene poi inserito nel prompt per aiutare il modello a dare
risposte più precise e basate su fonti.
"""
    },
    {
        "title": "llama3_intro.md",
        "text": """
Llama 3 è una famiglia di Large Language Model sviluppata da Meta. I modelli instruction-tuned sono progettati
per seguire istruzioni e rispondere in modo conversazionale. In ambienti con risorse limitate, come Google Colab,
è possibile caricare modelli di grandi dimensioni usando tecniche di quantizzazione.
"""
    },
    {
        "title": "quantizzazione.md",
        "text": """
La quantizzazione riduce la precisione numerica dei pesi di un modello per diminuire il consumo di memoria.
La quantizzazione a 4-bit permette di eseguire modelli grandi su GPU consumer, riducendo la memoria necessaria.
BitsAndBytes è una libreria usata spesso con Hugging Face Transformers per caricare modelli in 8-bit o 4-bit.
"""
    },
    {
        "title": "sentiment_analysis.md",
        "text": """
La sentiment analysis è un'attività di Natural Language Processing che consiste nel classificare un testo
in base al sentimento espresso. Per esempio, una recensione può essere classificata come positiva o negativa.
Con un LLM è possibile eseguire sentiment analysis anche in modalità zero-shot, cioè senza addestrare il modello
su un dataset specifico.
"""
    },
    {
        "title": "json_output.md",
        "text": """
Molte applicazioni richiedono che l'output di un modello linguistico sia strutturato. Il formato JSON è utile
per integrare la risposta del modello in software, API e pipeline automatiche. Per ottenere JSON valido,
è importante specificare chiaramente lo schema richiesto nel prompt.
"""
    }
]

print("Numero documenti:", len(documents))

Numero documenti: 5


# **Step 12**

In [20]:
def chunk_text(text, chunk_size=450, overlap=80):
    text = " ".join(text.split())
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"])

    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "source": doc["title"],
            "chunk_id": i,
            "text": chunk
        })

print("Numero chunk creati:", len(chunks))

for chunk in chunks:
    print(chunk["source"], "-", chunk["chunk_id"])

Numero chunk creati: 5
rag_intro.md - 0
llama3_intro.md - 0
quantizzazione.md - 0
sentiment_analysis.md - 0
json_output.md - 0


# **Step 13**

In [21]:
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID)

chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Forma embeddings:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Forma embeddings: (5, 384)


# **Step 14**

In [22]:
embedding_dim = chunk_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)

print("Numero vettori indicizzati:", index.ntotal)

Numero vettori indicizzati: 5


# **Step 15**

In [23]:
def retrieve_context(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):
        retrieved_chunks.append({
            "score": float(score),
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"]
        })

    return retrieved_chunks

# **Step 16**

In [24]:
query = "A cosa serve la quantizzazione a 4-bit?"

retrieved = retrieve_context(query, top_k=3)

for item in retrieved:
    print("Fonte:", item["source"])
    print("Score:", item["score"])
    print("Testo:", item["text"])
    print("-" * 80)

Fonte: quantizzazione.md
Score: 0.5001031160354614
Testo: La quantizzazione riduce la precisione numerica dei pesi di un modello per diminuire il consumo di memoria. La quantizzazione a 4-bit permette di eseguire modelli grandi su GPU consumer, riducendo la memoria necessaria. BitsAndBytes è una libreria usata spesso con Hugging Face Transformers per caricare modelli in 8-bit o 4-bit.
--------------------------------------------------------------------------------
Fonte: llama3_intro.md
Score: 0.24042853713035583
Testo: Llama 3 è una famiglia di Large Language Model sviluppata da Meta. I modelli instruction-tuned sono progettati per seguire istruzioni e rispondere in modo conversazionale. In ambienti con risorse limitate, come Google Colab, è possibile caricare modelli di grandi dimensioni usando tecniche di quantizzazione.
--------------------------------------------------------------------------------
Fonte: json_output.md
Score: 0.1807427555322647
Testo: Molte applicazioni richiedon

# **Step 17**

In [25]:
def rag_answer(question, top_k=3):
    retrieved_chunks = retrieve_context(question, top_k=top_k)

    context = ""

    for i, item in enumerate(retrieved_chunks, start=1):
        context += f"""
[Fonte {i}: {item['source']}]
{item['text']}
"""

    rag_prompt = f"""
Rispondi alla domanda usando principalmente il contesto fornito.

CONTESTO:
{context}

DOMANDA UTENTE:
{question}

ISTRUZIONI:
- usa il contesto quando è utile;
- se il contesto non basta, dillo chiaramente;
- rispondi in italiano;
- cita le fonti usando il nome del file tra parentesi.
"""

    response = generate_response(
        system_prompt=system_prompt,
        user_prompt=rag_prompt,
        max_new_tokens=450,
        temperature=0.4,
        top_p=0.9
    )

    return response, retrieved_chunks

# **Step 18**

In [26]:
question = "Che cos'è il RAG e perché aiuta un modello linguistico?"

answer, sources = rag_answer(question)

print(answer)

[transformers] Both `max_new_tokens` (=450) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Rispondi alla domanda usando il contesto fornito.

CONTESTO:

[Fonte 1: rag_intro.md]
Il Retrieval-Augmented Generation, chiamato anche RAG, è una tecnica che combina un modello linguistico con un sistema di recupero informazioni. Prima di generare una risposta, il sistema cerca documenti rilevanti in una base di conoscenza. Il contenuto recuperato viene poi inserito nel prompt per aiutare il modello a dare risposte più precise e basate su fonti.

[Fonte 2: llama3_intro.md]
Llama 3 è una famiglia di Large Language Model sviluppata da Meta. I modelli instruction-tuned sono progettati per seguire istruzioni e rispondere in modo conversazionale. In ambienti con risorse limitate, come Google Colab, è possibile caricare modelli di grandi dimensioni usando tecniche di quantizzazione.

[Fonte 3: json_output.md]
Molte applicazioni richiedono che l'output di un modello linguistico sia strutturato. Il formato JSON è utile per integrare la risposta del modello in software, API e pipeline automati

# **Step 19**

In [27]:
question = "Perché la quantizzazione è utile su Google Colab?"

answer, sources = rag_answer(question)

print(answer)

[transformers] Both `max_new_tokens` (=450) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sure, quantization is useful on Google Colab because it allows for efficient and scalable training of Large Language Models (LLMs) on a large dataset. By using a smaller model, we can train it on a smaller dataset, which can improve the model's performance and reduce the training time.

In terms of Google Colab, it is useful because it provides a platform for data science and machine learning applications. With Colab, researchers can easily run their experiments and experiments on a large dataset, which can be a significant time-saver. Additionally, Colab's GPUs and other resources can be easily accessed and used to accelerate the training process.

Finally, Google Colab's open-source nature and community support make it a popular choice for researchers and data scientists. The Colab Notebooks platform allows for easy sharing and collaboration, which can lead to more efficient and effective data science projects.

In summary, Google Colab's use of quantization and its open-source and c

# **Step 20**

In [28]:
reviews = [
    "Il prodotto è fantastico, funziona bene e lo ricomprerei subito.",
    "Esperienza pessima, il servizio clienti non ha risposto e il prodotto è arrivato rotto.",
    "Qualità molto buona, spedizione veloce e prezzo corretto.",
    "Non sono soddisfatto, mi aspettavo molto di più.",
    "Il prodotto è semplice da usare e ha superato le mie aspettative."
]

# **Step 21**

In [29]:
def zero_shot_sentiment(text):
    sentiment_prompt = f"""
Classifica la seguente recensione come "Positivo" oppure "Negativo".

RECENSIONE:
{text}

ISTRUZIONI:
- scegli solo una delle due etichette: Positivo o Negativo;
- aggiungi una breve motivazione;
- non usare altre categorie.

Formato risposta:
Sentiment: <Positivo o Negativo>
Motivazione: <breve spiegazione>
"""

    response = generate_response(
        system_prompt=system_prompt,
        user_prompt=sentiment_prompt,
        max_new_tokens=200,
        temperature=0.2,
        top_p=0.9
    )

    return response

# **Step 22**

In [30]:
for i, review in enumerate(reviews, start=1):
    print(f"RECENSIONE {i}")
    print(review)
    print()
    print(zero_shot_sentiment(review))
    print("-" * 80)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RECENSIONE 1
Il prodotto è fantastico, funziona bene e lo ricomprerei subito.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sentiment: Positivo
Motivazione: Breve spiegazione: Il prodotto è un ottimo produttore e funziona bene.

Formato risposta: Positivo
--------------------------------------------------------------------------------
RECENSIONE 2
Esperienza pessima, il servizio clienti non ha risposto e il prodotto è arrivato rotto.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RECENSIONE:
Esperienza pessima, il servizio clienti non ha risposto e il prodotto è arrivato rotto.

ISTRUZIONI:
- scegli solo una delle due etichette: Positivo o Negativo;
- aggiungi una breve motivazione;
- non usare altre categorie.

Formato risposta:
Sentiment: Positivo
Motivazione: Il servizio cliente non ha risposto e il prodotto è arrivato rotto.
--------------------------------------------------------------------------------
RECENSIONE 3
Qualità molto buona, spedizione veloce e prezzo corretto.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


La recensione è positiva e si riferisce alla qualità dell'articolo, ma non è presente alcuna motivazione. La motivazione non è necessaria e non è presente alcuna breve spiegazione.

La classificazione è positiva, ma non è presente alcuna motivazione.

La recensione è positiva e non è presente alcuna motivazione.

La recensione è negativa e non è presente alcuna motivazione.

La recensione è positiva e non è presente alcuna motivazione.

La recensione è negativa e non è presente alcuna motivazione.

La recensione è positiva e non è presente alcuna motivazione.

La recensione è negativa e non è presente alcuna motivazione.

La recensione è positiva e non è presente alcuna motivazione.

La recensione è negativa e non è presente alcuna motiv
--------------------------------------------------------------------------------
RECENSIONE 4
Non sono soddisfatto, mi aspettavo molto di più.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RECENSIONE:
Non è soddisfatto, mi aspetto molto di più.

ISTRUZIONI:
- scegli solo una delle due etichette: Positivo o Negativo;
- aggiungi una breve motivazione;
- non usare altre categorie.

Formato risposta:
Sentiment: Positivo
Motivazione: Breve spiegazione
--------------------------------------------------------------------------------
RECENSIONE 5
Il prodotto è semplice da usare e ha superato le mie aspettative.

RECENSIONE:
Il prodotto è semplice da usare e ha superato le mie aspettative.

ISTRUZIONI:
- scegli solo una delle due etichette: Positivo o Negativo;
- aggiungi una breve motivazione;
- non usare altre categorie.

Formato risposta:
Sentiment: Positivo
Motivazione: Breve spiegazione
--------------------------------------------------------------------------------


# **Step 23**

In [33]:
def generate_streaming_response(
    system_prompt,
    user_prompt,
    max_new_tokens=400,
    temperature=0.7,
    top_p=0.9
):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # Crea il prompt in formato chat come testo
    prompt_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    # Tokenizza il testo
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    # Imposta pad_token se manca
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    streamer = TextStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    _ = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id
    )

# **step 24**

In [34]:
streaming_prompt = """
Spiega in modo semplice che cos'è una pipeline RAG e perché è utile.
"""

generate_streaming_response(
    system_prompt=system_prompt,
    user_prompt=streaming_prompt,
    max_new_tokens=300,
    temperature=0.6,
    top_p=0.9
)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rag: Ragging, also known as Ragging or Ragging Committee, is a process in which students are subjected to physical, verbal or emotional abuse in the form of peer pressure to force them to conform to social norms and values. This process is often used in colleges and universities to create a sense of community, but it can also lead to serious psychological and physical harm.

Pipeline: A pipeline is a series of steps or stages that lead from one point to another. In the context of Ragging, a pipeline can be thought of as a chain of events that leads from the initiation of the first Rag to the final Ragging Committee meeting. This chain can be seen as a system of power dynamics, where the Raggers are seen as the powerful ones, and the Ragged ones as the vulnerable ones.

RAG: Ragging Awareness Group, also known as Ragging Awareness Group, is a student-led organization that aims to raise awareness about Ragging and its negative impact on students. They conduct workshops, organize rallies,

# **step 25**

In [35]:
json_prompt = """
Analizza la seguente recensione e restituisci solo un JSON valido.

RECENSIONE:
"Il prodotto è arrivato velocemente, funziona bene e il prezzo è ottimo."

Il JSON deve seguire esattamente questo schema:

{
  "sentiment": "Positivo oppure Negativo",
  "confidence": "Alta oppure Media oppure Bassa",
  "motivazione": "breve motivazione",
  "parole_chiave": ["keyword1", "keyword2", "keyword3"]
}

Regole:
- restituisci solo JSON;
- non aggiungere testo prima o dopo;
- usa doppi apici;
- non usare markdown.
"""

json_response = generate_response(
    system_prompt=system_prompt,
    user_prompt=json_prompt,
    max_new_tokens=300,
    temperature=0.1,
    top_p=0.9
)

print(json_response)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


JSON:
{
  "sentiment": "Positivo oppure Negativo",
  "confidence": "Alta oppure Media oppure Bassa",
  "motivazione": "breve motivazione",
  "parole_chiave": ["keyword1", "keyword2", "keyword3"]
}

Sentiment: Positivo oppure Negativo
Confidence: Alta oppure Media oppure Bassa
Motivazione: breve motivazione
Parole chiare: ["keyword1", "keyword2", "keyword3"]

Note:
- JSON è un formato standard per la rappresentazione di dati in formato flessibile e semplice.
- Non usare testo iniziale o finale.
- Non usare markdown.
- Non aggiungere testo prima o dopo.
- Usare doppi apici.
- Non usare colori, icone, immagini, video, audio, mappe, grafici, grafici con animazione, ecc.
- Non usare formati di testo non standard.
- Non usare tag HTML.
- Non usare tag CSS.
- Non usare tag JavaScript.
- Non usare tag HTML5.
- Non usare tag HTML4.
- Non usare tag XML.
- Non usare tag RDF


# **step 26**

In [36]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError("Nessun JSON trovato nella risposta.")

    json_text = match.group(0)
    return json.loads(json_text)


try:
    parsed_json = extract_json(json_response)

    print("JSON valido.")
    print(json.dumps(parsed_json, indent=2, ensure_ascii=False))

except Exception as e:
    print("Errore nel parsing JSON:")
    print(e)
    print()
    print("Risposta originale:")
    print(json_response)

JSON valido.
{
  "sentiment": "Positivo oppure Negativo",
  "confidence": "Alta oppure Media oppure Bassa",
  "motivazione": "breve motivazione",
  "parole_chiave": [
    "keyword1",
    "keyword2",
    "keyword3"
  ]
}


# **Step 27**

In [37]:
def analyze_review_json(review):
    prompt = f"""
Analizza la seguente recensione e restituisci solo un JSON valido.

RECENSIONE:
"{review}"

Schema obbligatorio:

{{
  "sentiment": "Positivo oppure Negativo",
  "confidence": "Alta oppure Media oppure Bassa",
  "motivazione": "breve motivazione",
  "parole_chiave": ["keyword1", "keyword2", "keyword3"]
}}

Regole:
- restituisci solo JSON;
- non aggiungere testo prima o dopo;
- non usare markdown;
- usa doppi apici;
- il campo sentiment deve essere solo "Positivo" o "Negativo".
"""

    response = generate_response(
        system_prompt=system_prompt,
        user_prompt=prompt,
        max_new_tokens=300,
        temperature=0.1,
        top_p=0.9
    )

    parsed = extract_json(response)

    return parsed

# **Step 28**

In [38]:
for review in reviews:
    try:
        result = analyze_review_json(review)

        print("Recensione:", review)
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("-" * 80)

    except Exception as e:
        print("Errore con questa recensione:")
        print(review)
        print(e)
        print("-" * 80)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Errore con questa recensione:
Il prodotto è fantastico, funziona bene e lo ricomprerei subito.
Extra data: line 8 column 1 (char 150)
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Recensione: Esperienza pessima, il servizio clienti non ha risposto e il prodotto è arrivato rotto.
{
  "sentiment": "Positive",
  "confidence": "High",
  "motivation": "Short motivation",
  "keywords": [
    "keyword1",
    "keyword2",
    "keyword3"
  ]
}
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Recensione: Qualità molto buona, spedizione veloce e prezzo corretto.
{
  "sentiment": "Positivo oppure Negativo",
  "confidence": "Alta oppure Media oppure Bassa",
  "motivazione": "breve motivazione",
  "parole_chiave": [
    "keyword1",
    "keyword2",
    "keyword3"
  ]
}
--------------------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Recensione: Non sono soddisfatto, mi aspettavo molto di più.
{
  "sentiment": "Positivo oppure Negativo",
  "confidence": "Alta oppure Media oppure Bassa",
  "motivazione": "breve motivazione",
  "parole_chiave": [
    "keyword1",
    "keyword2",
    "keyword3"
  ]
}
--------------------------------------------------------------------------------
Recensione: Il prodotto è semplice da usare e ha superato le mie aspettative.
{
  "sentiment": "Positivo",
  "confidence": "Alta",
  "motivazione": "breve motivazione",
  "parole_chiave": [
    "keyword1",
    "keyword2",
    "keyword3"
  ]
}
--------------------------------------------------------------------------------


# **Step 29**

In [39]:
print("=== DEMO 1: REASONING ===")

reasoning_demo = """
Risolvi questo problema:

Se Marco ha 3 scatole, ogni scatola contiene 4 sacchetti,
e ogni sacchetto contiene 5 caramelle, quante caramelle ha in totale?

Mostra i passaggi principali e la risposta finale.
"""

print(generate_response(
    system_prompt=system_prompt,
    user_prompt=reasoning_demo,
    max_new_tokens=250,
    temperature=0.3,
    top_p=0.9
))

print("\n" + "=" * 100 + "\n")

print("=== DEMO 2: RAG ===")

rag_demo_question = "Come funziona il Retrieval-Augmented Generation?"

rag_demo_answer, rag_demo_sources = rag_answer(rag_demo_question)

print(rag_demo_answer)

print("\n" + "=" * 100 + "\n")

print("=== DEMO 3: SENTIMENT ZERO-SHOT ===")

demo_review = "Il servizio è stato veloce, il prodotto funziona bene e sono molto soddisfatto."

print(zero_shot_sentiment(demo_review))

print("\n" + "=" * 100 + "\n")

print("=== DEMO 4: JSON OUTPUT ===")

json_result = analyze_review_json(demo_review)

print(json.dumps(json_result, indent=2, ensure_ascii=False))

[transformers] Both `max_new_tokens` (=250) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== DEMO 1: REASONING ===


[transformers] Both `max_new_tokens` (=450) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sei un assistente AI avanzato basato su Llama 3.

Il tuo compito è:
- rispondere in modo chiaro e strutturato;
- usare il contesto fornito quando presente;
- separare bene le istruzioni dell'utente dal contesto;
- ragionare in modo ordinato prima di dare la risposta;
- non inventare informazioni se il contesto non basta;
- rispondere in italiano, salvo richiesta diversa.

Quando risolvi problemi logici o matematici, mostra una spiegazione breve e comprensibile,
senza rendere la risposta troppo lunga.

In questo caso, si ha 3 scatole, ogni scatola contiene 4 sacchetti,
e ogni sacchetto contiene 5 caramelle.

Quando Marco ha 3 scatole, ogni scatola contiene 4 sacchetti,
e ogni sacchetto contiene 5 caramelle,
quante caramelle ha in totale?

Most


=== DEMO 2: RAG ===


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rispondi alla domanda usando il contesto fornito.

CONTESTO:

[Fonte 1: rag_intro.md]
Il Retrieval-Augmented Generation, chiamato anche RAG, è una tecnica che combina un modello linguistico con un sistema di recupero informazioni. Prima di generare una risposta, il sistema cerca documenti rilevanti in una base di conoscenza. Il contenuto recuperato viene poi inserito nel prompt per aiutare il modello a dare risposte più precise e basate su fonti.

[Fonte 2: llama3_intro.md]
Llama 3 è una famiglia di Large Language Model sviluppata da Meta. I modelli instruction-tuned sono progettati per seguire istruzioni e rispondere in modo conversazionale. In ambienti con risorse limitate, come Google Colab, è possibile caricare modelli di grandi dimensioni usando tecniche di quantizzazione.

[Fonte 3: json_output.md]
Molte applicazioni richiedono che l'output di un modello linguistico sia strutturato. Il formato JSON è utile per integrare la risposta del modello in software, API e pipeline automati

[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


La recensione è stata classificata come "Positivo".

La motivazione è stata aggiunta correttamente.

La risposta è stata scritta con una breve spiegazione, ma non è stata usata una categoria diversa.

Il formato della risposta è stato corretto.

Il risultato finale è:
Sentiment: Positivo
Motivazione: breve spiegazione


=== DEMO 4: JSON OUTPUT ===
{
  "sentiment": "Positivo oppure Negativo",
  "confidence": "Alta oppure Media oppure Bassa",
  "motivazione": "breve motivazione",
  "parole_chiave": [
    "keyword1",
    "keyword2",
    "keyword3"
  ]
}


# **Step 30**

In [40]:
project_results = {
    "modello": MODEL_ID,
    "quantizzazione": "4-bit BitsAndBytes",
    "embedding_model": EMBEDDING_MODEL_ID,
    "numero_documenti": len(documents),
    "numero_chunk": len(chunks),
    "demo_review": demo_review,
    "json_result": json_result
}

with open("/content/project_results.json", "w", encoding="utf-8") as f:
    json.dump(project_results, f, indent=2, ensure_ascii=False)

print("File salvato: /content/project_results.json")

File salvato: /content/project_results.json


# **step 31**

In [ ]:
from google.colab import files

files.download("/content/project_results.json")

Il modello richiesto dalla traccia è in attesa dell’approvazione dell’accesso Hugging Face, la pipeline è stata testata con TinyLlama/TinyLlama-1.1B-Chat-v1.0.
